# QBioCode-ADMET Benchmark Tutorial

This tutorial walks through the full ADMET benchmark pipeline:
1. Downloading and preparing TDC ADMET datasets
2. Running QProfiler on a single endpoint (hERG)
3. Training QSage on benchmark results
4. Loading best checkpoints and running test inference

**Install dependencies:**
```bash
pip install 'qbiocode[admet]'
```

In [ ]:
import os, sys
# Point to repo root if running from tutorial directory
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

import qbiocode as qbc
print('QBioCode version:', qbc.__version__)

## Step 1: Prepare a Single ADMET Endpoint (hERG)

In [ ]:
from qbiocode.data_adapters import TDCAdmetLoader, MolecularFeaturizer

# Download and binarize the hERG endpoint
loader = TDCAdmetLoader(data_dir='data/admet_tutorial', qml_train_cap=200, seed=42)
meta = loader.prepare_endpoint('hERG')
print('Endpoint metadata:', meta)

In [ ]:
# Featurize with ECFP4
feat = MolecularFeaturizer(featurizer='ecfp4')
feat.featurize_endpoint('data/admet_tutorial/hERG')

import pandas as pd
train_df = pd.read_csv('data/admet_tutorial/hERG/ecfp4/train.csv')
print(f'Train shape: {train_df.shape}  |  Class balance: {train_df["Y"].mean():.3f}')

## Step 2: Run QProfiler on hERG / ECFP4

In [ ]:
# Use the CLI (recommended) or the Python API below
# From repo root:
# qprofiler --config-dir=qbiocode/apps/qprofiler/configs --config-name=admet_config \
#           folder_path=data/admet_tutorial/hERG/ecfp4 \
#           model=[svc,rf,qsvc] iter=2 backend=simulator
print('Run the qprofiler CLI command above from the repo root.')

## Step 3: Load Checkpoints and Infer on Test Set

In [ ]:
# After a QProfiler run with checkpoint_dir set in the config:
from qbiocode.utils.model_checkpoint import get_best_index, infer_from_best
import numpy as np

CHECKPOINT_DIR = 'results/admet_tutorial/checkpoints'

# Show what was saved
index = get_best_index(CHECKPOINT_DIR)
for dataset_key, models in index.items():
    for model_name, info in models.items():
        print(f'  {dataset_key:40s} / {model_name:8s}  val_f1={info["val_f1"]:.4f}')

In [ ]:
# Load test data
test_df = pd.read_csv('data/admet_tutorial/hERG/ecfp4/test.csv')
X_test = test_df.iloc[:, :-1].to_numpy()
y_test = test_df.iloc[:, -1].to_numpy().astype(int)

# Infer with best RF checkpoint
from sklearn.metrics import roc_auc_score
y_pred = infer_from_best('rf', 'hERG_ecfp4', X_test, CHECKPOINT_DIR)
print(f'hERG / ECFP4 / RF  test AUROC = {roc_auc_score(y_test, y_pred):.4f}')

## Next Steps

- Run the full sweep: `bash experiments/admet_benchmark/02_run_qprofiler_admet.sh`
- Train QSage: `python experiments/admet_benchmark/03_train_qsage_admet.py`
- Generate paper figures: open `experiments/admet_benchmark/05_analysis.ipynb`

See `paper_plan.md` at the repo root for the complete research plan.